In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen

In [ ]:
import pandas as pd
df = pd.read_csv('L5-Long_mu=0.005.csv', sep=';')
# Функция для проверки и преобразования строки в float (с заменой запятой на точку)
def convert_to_float(value):
    if type(value) == str:
        return float(value.replace(',', '.'))
    else:
        return value

for C in df.columns[:19]:
    df[C] = df[C].apply(lambda x: convert_to_float(x))

In [ ]:
df.head()

In [ ]:
plt.plot(df['a'], df['x1'], label='x1')
plt.plot(df['a'], df['x2'], label='x2')
plt.grid()
plt.xlabel('a')
plt.ylabel('x')
plt.legend()
plt.show()

In [ ]:
plt.plot(df['a'][0:-1], df['v1'][0:-1], label='v1')
plt.plot(df['a'][0:-1], df['v2'][0:-1], label='v2')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('v')
plt.show()


In [ ]:
plt.plot(df['a'][0:-1], df['Rho_1'][0:-1], label='Rho_1')
plt.plot(df['a'][0:-1], df['Rho_1_i'][0:-1], label='Rho_1_i')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('Rho')
plt.show()


In [ ]:
ax = plt.figure().add_subplot(projection='3d')
ax.scatter(df['a'][0:-1], df['Rho_1'][0:-1], df['Rho_1_i'][0:-1], label='Rho_1')
ax.scatter(df['a'][0:-1], df['Rho_2'][0:-1], df['Rho_2_i'][0:-1], label='Rho_2')
ax.scatter(df['a'][0:-1], df['Rho_3'][0:-1], df['Rho_3_i'][0:-1], label='Rho_3')
ax.scatter(df['a'][0:-1], df['Rho_4'][0:-1], df['Rho_4_i'][0:-1], label='Rho_4')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('Rho_4')
plt.show()

In [ ]:
plt.plot(df['a'][0:-1], df['T'][0:-1])
plt.grid()
plt.xlabel('a')
plt.ylabel('T')
plt.show()

In [ ]:
plt.plot(df['Jacobi Const'][0:-1], df['T'][0:-1])
plt.grid()
plt.xlabel('J')
plt.ylabel('T')
plt.show()

In [ ]:
df['a'][np.argmax(df['Jacobi Const'])]

In [ ]:
plt.plot(df['a'][0:45], df['Jacobi Const'][0:45])
plt.grid()
plt.xlabel('a')
plt.ylabel('J')
plt.show()

In [ ]:
plt.plot(df['a'][45:-1], df['Jacobi Const'][45:-1])
plt.grid()
plt.xlabel('a')
plt.ylabel('J')
plt.show()

In [ ]:
model = PySRRegressor(
        niterations=500,  # < Increase me for better results
        binary_operators=["*", "+", "-"],
        unary_operators=[
            'exp',
            #'cos',
            #'sin'
            'log',
        ],
        #extra_sympy_mappings={"pimult": lambda x: np.pi*x},
        #nested_constraints={"cos": {"cos": 0, "sin": 0},"sin": {"cos": 0, "sin": 0}},
        #constraints={'^': (-1, 1)},
        #nested_constraints={"exp": {"exp": 0}},
        nested_constraints={"exp": {"exp": 0},"log": {"log": 0}},
        populations=36,
        verbosity=0,
        #batching=True,
        warm_start = False,
        #denoise=True
    )

In [ ]:
input_values = np.expand_dims(np.array(df['a'][45:-1]).flatten(),-1)
target_values = np.expand_dims(np.array(df['T'][45:-1]).flatten(),-1)
model.fit(input_values, target_values,
        variable_names=["a"])

In [ ]:
from sympy.utilities.lambdify import lambdastr
from sympy.abc import a

In [ ]:
N=None

test_preds = model.predict(input_values,N)
test_trues = target_values.reshape(test_preds.shape)
MSE = np.square(np.subtract(test_trues,test_preds)).mean() 
s = model.sympy(N).simplify()
display(s)
RMSE = np.sqrt(MSE)
print(RMSE)

print('Python Code:')
print(lambdastr(a, s))
plt.plot(input_values, test_trues)
plt.plot(input_values, test_preds,'--')
plt.grid()
plt.show()